# YOLO11-OBB passport training on Colab

**Runtime:** Runtime > Change runtime type > **T4 GPU**. Ultralytics does not train on TPU.

**Before running:** upload your dataset zip (any name, e.g. `data.zip`) to `MyDrive/yolo11_obb/`.
The zip must contain `data.yaml`, `images/` and `labels/` (with `train` and `val` subfolders).

In [ ]:
# 1) Check the runtime
import os, torch
if "COLAB_TPU_ADDR" in os.environ:
    raise SystemExit("TPU runtime detected. Switch to GPU: Runtime > Change runtime type > T4 GPU")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Install Ultralytics
!pip install -q ultralytics pyyaml

In [ ]:
# 3) Mount Drive, find the zip, extract it to /content/raw and list what it contains
import glob, pathlib, subprocess, os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
zips = sorted(glob.glob('/content/drive/MyDrive/yolo11_obb/*.zip'))
if not zips:
    raise SystemExit("No .zip found in MyDrive/yolo11_obb/. Upload your dataset zip there.")
zip_path = zips[0]
print("zip:", zip_path)
EXTRACT = pathlib.Path('/content/raw')
subprocess.run(['unzip', '-q', '-o', zip_path, '-d', str(EXTRACT)], check=True)
print("extracted top level:", sorted(p.name for p in EXTRACT.iterdir()))

In [ ]:
%%writefile build_obb_dataset.py
"""Build a leak-free YOLO11-OBB dataset from a flat folder of passport images and labels.

Input (flat layout, as in Data_aug):
    <src>/images/*.jpg
    <src>/labels/*.txt          same stem as the image

Label formats handled (normalized to [0,1]):
    - 5 values : class cx cy w h            -> converted to 4 corners (axis-aligned OBB)
    - 9 values : class x1 y1 ... x4 y4      -> kept as is (already oriented)

Output (YOLO-OBB layout):
    <dst>/images/{train,val}/*.jpg
    <dst>/labels/{train,val}/*.txt         one line per object: class x1 y1 x2 y2 x3 y3 x4 y4
    <dst>/data.yaml

Split: all augmentations of one base photo (base_augN.jpg) go to the SAME split,
so an augmented copy in train never shares its original with val.

For a folder that is already split (src/train/... and src/val/...), pass --presplit
and the existing split is kept.
"""
import argparse
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

NAMES = {i: str(i) for i in range(15)}  # replace with the real field names once confirmed
AUG_RE = re.compile(r"_aug\d+$")
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def base_of(stem: str) -> str:
    return AUG_RE.sub("", stem)


def to_obb_line(parts):
    cls = int(float(parts[0]))
    vals = [float(v) for v in parts[1:]]
    if len(vals) == 4:  # cx cy w h -> 4 corners, clockwise from top-left
        cx, cy, w, h = vals
        x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
        pts = [x1, y1, x2, y1, x2, y2, x1, y2]
    elif len(vals) == 8:
        pts = vals
    else:
        raise ValueError(f"unsupported label with {len(vals)} coords: {parts}")
    pts = [min(max(p, 0.0), 1.0) for p in pts]  # keep inside the image
    return f"{cls} " + " ".join(f"{p:.6f}" for p in pts)


def read_label(path: Path):
    lines = []
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = raw.split()
        if parts:
            lines.append(to_obb_line(parts))
    return lines


def collect(src: Path):
    """Return {stem: (image_path, [obb_lines], split_or_None)} for all labelled images."""
    items = {}
    presplit = (src / "train").is_dir() and (src / "val").is_dir()
    dirs = [(s, src / s) for s in ("train", "val")] if presplit else [(None, src)]
    for split, root in dirs:
        img_dir, lab_dir = root / "images", root / "labels"
        if not img_dir.is_dir() or not lab_dir.is_dir():
            raise SystemExit(f"missing folder: {img_dir} or {lab_dir}")
        images = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT}
        for lab in sorted(lab_dir.glob("*.txt")):
            if lab.stem not in images:
                print(f"WARNING: no image for label {lab.name}, skipped")
                continue
            items[lab.stem] = (images[lab.stem], read_label(lab), split)
    return items, presplit


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, type=Path)
    ap.add_argument("--dst", required=True, type=Path)
    ap.add_argument("--val-frac", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--presplit", action="store_true", help="keep an existing train/val split from src")
    args = ap.parse_args()

    items, presplit = collect(args.src)
    if not items:
        raise SystemExit(f"no labelled images found in {args.src}")

    if presplit and args.presplit:
        split_of = {s: items[s][2] for s in items}
        print("using existing train/val split (--presplit)")
    else:
        # group augmentations by base photo, then split the groups
        groups = defaultdict(list)
        for s in items:
            groups[base_of(s)].append(s)
        keys = sorted(groups)
        random.Random(args.seed).shuffle(keys)
        n_val = max(1, round(len(keys) * args.val_frac))
        val_keys = set(keys[:n_val])
        split_of = {s: ("val" if base_of(s) in val_keys else "train") for s in items}

    # sanity: no base photo in both splits
    by_base = defaultdict(set)
    for s in items:
        by_base[base_of(s)].add(split_of[s])
    shared = [b for b, sp in by_base.items() if len(sp) > 1]
    if shared:
        raise SystemExit(f"leakage: base photos in both splits: {shared[:5]}")

    if args.dst.exists():
        shutil.rmtree(args.dst)
    for sp in ("train", "val"):
        (args.dst / "images" / sp).mkdir(parents=True)
        (args.dst / "labels" / sp).mkdir(parents=True)

    cls_count = {"train": Counter(), "val": Counter()}
    for s, (img_path, lines, _) in items.items():
        sp = split_of[s]
        shutil.copy2(img_path, args.dst / "images" / sp / img_path.name)
        (args.dst / "labels" / sp / f"{s}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        for ln in lines:
            cls_count[sp][int(ln.split()[0])] += 1

    names_block = "\n".join(f"  {i}: {NAMES[i]}" for i in sorted(NAMES))
    (args.dst / "data.yaml").write_text(
        f"path: {args.dst.resolve().as_posix()}\n"
        f"train: images/train\nval: images/val\n"
        f"names:\n{names_block}\n",
        encoding="utf-8",
    )

    n_tr = sum(v == "train" for v in split_of.values())
    n_va = sum(v == "val" for v in split_of.values())
    print(f"base photos: {len(by_base)} | train images: {n_tr} | val images: {n_va}")
    for sp in ("train", "val"):
        print(f"{sp} objects per class:", dict(sorted(cls_count[sp].items())))
    missing_val = [c for c in NAMES if cls_count["val"][c] == 0]
    if missing_val:
        print("WARNING: classes absent from val:", missing_val, "-> try another --seed")


if __name__ == "__main__":
    main()


In [ ]:
# 4) Find a ready dataset (data.yaml) or build one from raw images/ + labels/
import pathlib, subprocess
DATASET = pathlib.Path('/content/dataset_obb')
yamls = sorted(EXTRACT.rglob('data.yaml'))
if yamls:
    DATA_ROOT = yamls[0].parent
    print("ready dataset found:", DATA_ROOT)
else:
    cands = [p.parent for p in EXTRACT.rglob('labels') if p.is_dir() and (p.parent/'images').is_dir()]
    if not cands:
        raise SystemExit("No data.yaml and no images/ + labels/ folders in the zip. Check the zip contents above.")
    src = cands[0]
    print("no data.yaml: building OBB dataset from raw folder:", src)
    subprocess.run(['python', 'build_obb_dataset.py', '--src', str(src), '--dst', str(DATASET)], check=True)
    DATA_ROOT = DATASET
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
# 4) Fix data.yaml path for Colab and check image/label counts
import yaml
yaml_path = DATA_ROOT/'data.yaml'
cfg = yaml.safe_load(yaml_path.read_text())
cfg['path'] = str(DATA_ROOT)
yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
for sp in ('train','val'):
    n_img = len([p for p in (DATA_ROOT/'images'/sp).iterdir()])
    n_lab = len(list((DATA_ROOT/'labels'/sp).glob('*.txt')))
    print(sp, 'images', n_img, 'labels', n_lab)
    assert n_img == n_lab > 0, f'{sp}: image/label mismatch'

In [ ]:
# 5) Train. Checkpoints go to Drive so they survive a disconnect.
from ultralytics import YOLO
RUN_DIR = '/content/drive/MyDrive/yolo11_obb/runs'
last = pathlib.Path(RUN_DIR)/'passport_yolo11_obb'/'weights'/'last.pt'
if last.exists():
    model = YOLO(str(last)); model.train(resume=True)
else:
    model = YOLO('yolo11s-obb.pt')
    model.train(
        data=str(yaml_path),
        epochs=100, imgsz=640, batch=16, device=0, workers=2,
        patience=30, degrees=10.0, close_mosaic=10,
        seed=42, deterministic=True, plots=True,
        project=RUN_DIR, name='passport_yolo11_obb', exist_ok=True,
    )

In [ ]:
# 6) Validate the best checkpoint and export ONNX to Drive
best = YOLO(str(pathlib.Path(RUN_DIR)/'passport_yolo11_obb'/'weights'/'best.pt'))
m = best.val(data=str(yaml_path), split='val', imgsz=640)
print('VAL mAP50-95(B):', round(m.box.map, 4), '| mAP50:', round(m.box.map50, 4))
onnx = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
import shutil; shutil.copy(onnx, '/content/drive/MyDrive/yolo11_obb/passport_obb.onnx')
print('saved to Drive: passport_obb.onnx')

In [ ]:
# 7) Save best model and Excel report to Drive
import shutil
from pathlib import Path

import pandas as pd
from openpyxl.chart import LineChart, Reference

MAP50 = "metrics/mAP50(B)"
MAP5095 = "metrics/mAP50-95(B)"
PREC = "metrics/precision(B)"
REC = "metrics/recall(B)"


def _add_chart(ws, df, title, cols, anchor, y_title):
    chart = LineChart()
    chart.title = title
    chart.x_axis.title = "epoch"
    chart.y_axis.title = y_title
    chart.height, chart.width = 8, 16
    epoch_col = list(df.columns).index("epoch") + 1
    n = len(df) + 1  # header row + data rows
    for name in cols:
        col = list(df.columns).index(name) + 1
        data = Reference(ws, min_col=col, min_row=1, max_row=n)
        chart.add_data(data, titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=epoch_col, min_row=2, max_row=n))
    ws.add_chart(chart, anchor)


def export_results(run_dir, out_dir):
    run_dir, out_dir = Path(run_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    best_pt = run_dir / "weights" / "best.pt"
    if best_pt.exists():
        shutil.copy2(best_pt, out_dir / "best.pt")
        print("saved:", out_dir / "best.pt")
    else:
        print("WARNING: best.pt not found in", run_dir / "weights")

    csv_path = run_dir / "results.csv"
    if not csv_path.exists():
        raise SystemExit(f"results.csv not found in {run_dir}")
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]

    fitness = 0.1 * df[MAP50] + 0.9 * df[MAP5095]
    b = int(fitness.idxmax())
    last = df.iloc[-1]
    best = df.iloc[b]
    summary = pd.DataFrame([
        ("best epoch (by fitness)", int(best["epoch"])),
        ("mAP50-95 at best epoch", round(float(best[MAP5095]), 4)),
        ("mAP50 at best epoch", round(float(best[MAP50]), 4)),
        ("precision at best epoch", round(float(best[PREC]), 4)),
        ("recall at best epoch", round(float(best[REC]), 4)),
        ("epochs completed", int(last["epoch"])),
        ("final mAP50-95", round(float(last[MAP5095]), 4)),
        ("total training time (min)", round(float(last["time"]) / 60, 1)),
        ("best.pt copied to", str(out_dir / "best.pt")),
    ], columns=["item", "value"])

    xlsx = out_dir / "training_results.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="per_epoch", index=False)
        summary.to_excel(writer, sheet_name="summary", index=False)
        ws = writer.sheets["per_epoch"]
        ws.freeze_panes = "B2"
        for col_cells in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max(width + 2, 10), 28)
        far = len(df.columns) + 2
        from openpyxl.utils import get_column_letter as L
        _add_chart(ws, df, "mAP per epoch", [MAP50, MAP5095], f"{L(far)}2", "mAP")
        _add_chart(ws, df, "Precision and recall per epoch", [PREC, REC], f"{L(far)}20", "value")
        _add_chart(ws, df, "Training losses", ["train/box_loss", "train/cls_loss", "train/dfl_loss"],
                   f"{L(far)}38", "loss")
        _add_chart(ws, df, "Validation losses", ["val/box_loss", "val/cls_loss", "val/dfl_loss"],
                   f"{L(far)}56", "loss")
        writer.sheets["summary"].column_dimensions["A"].width = 30
        writer.sheets["summary"].column_dimensions["B"].width = 50

    print("saved:", xlsx)
    print(summary.to_string(index=False))
    return xlsx

final_dir = pathlib.Path(RUN_DIR)/'passport_yolo11_obb'/'final'
export_results(run_dir=pathlib.Path(RUN_DIR)/'passport_yolo11_obb', out_dir=final_dir)
print('Final files in Drive:', final_dir)